In [0]:
USE CATALOG students_data;
USE SCHEMA `git-happens-schema`;

In [0]:
SELECT * FROM bronze_taxi LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE fact_trips AS
SELECT
  Booking_ID,
  Driver,
  Vehicle,
  Pickup_Zone,
  Destination_Zone,
  Source,
  Pickup_Due,
  Completed,
  Payment_Type,
  Priority,
  Capabilities,
  Booking_source,
  Price,
  Distance,
  Time_Dispatched,
  Time_Vehicle_Arrived,
  Time_Picked_Up,
  Pickup_Latitude,
  Pickup_Longitude,
  Destination_Latitude,
  Destination_Longitude,
  Booked_by,
  timestamp AS ingestion_timestamp
FROM bronze_taxi;



In [0]:
SELECT COUNT(*) AS total_rows FROM fact_trips;
-- SELECT * FROM fact_trips LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE silver_taxi AS
SELECT
  Booking_ID,
  CAST(REPLACE(Driver, '#', '') AS INT) AS Driver,
  Vehicle,
  Pickup_Zone,
  Destination_Zone,
  Source AS Status,
  TO_TIMESTAMP(Pickup_Due, 'dd/MM/yyyy HH:mm') AS Pickup_Due,
  TO_TIMESTAMP(Completed, 'dd/MM/yyyy HH:mm') AS Completed,
  Payment_Type,
  Priority,
  ARRAY_JOIN(
    TRANSFORM(
      FILTER(SPLIT(Capabilities, ''), c -> c != ''),
      c -> CASE c
        WHEN 'Z' THEN 'Card Reader'
        WHEN 'D' THEN 'Delivery'
        WHEN 'H' THEN 'High Car'
        WHEN 'L' THEN 'Low Car'
        WHEN 'W' THEN 'Wheelchair'
        WHEN 'M' THEN 'Minibus'
        WHEN 'F' THEN 'Female'
        WHEN 'V' THEN 'VIP'
        WHEN 'T' THEN 'Tour'
        WHEN 'P' THEN 'Pet'
        WHEN '6' THEN '6 Seater'
        WHEN '7' THEN '7 Seater'
        WHEN '8' THEN '8 Seater'
        ELSE 'Other'
      END
    ),
    ', '
  ) AS Capabilities,
  REPLACE (Booking_source,'(Web)','') AS Booking_source,
  Price,
  Distance,
  TO_TIMESTAMP(Time_Dispatched, 'dd/MM/yyyy HH:mm') AS Time_Dispatched,
  TO_TIMESTAMP(Time_Vehicle_Arrived, 'dd/MM/yyyy HH:mm') AS Time_Vehicle_Arrived,
  TO_TIMESTAMP(Time_Picked_Up, 'dd/MM/yyyy HH:mm') AS Time_Picked_Up,
  Pickup_Latitude,
  Pickup_Longitude,
  Destination_Latitude,
  Destination_Longitude,
  Booked_by,
  timestamp AS ingestion_timestamp
FROM bronze_taxi;

In [0]:
SELECT * from silver_taxi

In [0]:
-- Confirm no hashtag drivers remain and all are integers
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT Driver) AS distinct_drivers,
  SUM(CASE WHEN Driver IS NULL THEN 1 ELSE 0 END) AS null_drivers
FROM silver_taxi;